<a href="https://colab.research.google.com/github/Luigi2908/Segmentaci-n-e-commerce/blob/main/CLTV_Prediction_(_Proyecto_3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
datacertlaboratoria_proyecto_3_segmentacin_de_clientes_en_ecommerce_path = kagglehub.dataset_download('datacertlaboratoria/proyecto-3-segmentacin-de-clientes-en-ecommerce')

print('Data source import complete.')


In [ ]:
# CLTV Prrdiction in E-commerce
!pip install Lifetimes==0.2.2.2
# !pip install scipy==1.1.0


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.special

pd.set_option('display.float_format', lambda x: '%.2f' % x)

test = pd.read_csv("../input/proyecto-3-segmentacin-de-clientes-en-ecommerce/ventas-por-factura.csv")
df = test.copy()
df.head()

In [ ]:
# N° de factura    : Invoice_ID
# Fecha de factura : Invoice_Date
# ID Cliente       : Customer_ID
# País             : Country
# Cantidad         : Quantity
# Monto            : Amount


df.columns = ["Invoice_ID","Invoice_Date", "Customer_ID","Country",
            "Quantity","Amount"]
df.head()

In [ ]:
df.info()

# separate the date variable from time
df["Invoice_Date"] = pd.to_datetime(df["Invoice_Date"]).dt.date
# convert object Invoice_Date type date to datetime
df["Invoice_Date"] = pd.to_datetime(df["Invoice_Date"])

# convert object Amount type to float

df["Amount"] = df["Amount"].str.replace(",",".").astype(float)


df.info()

In [ ]:
df.isnull().sum()
df=df.dropna()
df.isnull().sum()

In [ ]:
df.describe().T

In [ ]:
#take out the return invoice from df   ( # C570727)

df.head()
df = df[~df["Invoice_ID"].str.contains("C")]

df.describe().T

In [ ]:
df.head()
df[df["Amount"] == 0.0].shape

# take out the Amount zero values from df
df = df[df["Amount"] > 0.0]

In [ ]:

def grab_col_names(dataframe, cat_th=10, car_th=20):


 # cat_cols, cat_but_car

    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # num_cols
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')
    return cat_cols, num_cols, cat_but_car


cat_cols, num_cols, cat_but_car = grab_col_names(df)

In [ ]:

def outlier_thresholds(dataframe, variable):
    quartile1 = dataframe[variable].quantile(0.01)
    quartile3 = dataframe[variable].quantile(0.99)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit


def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

for col in num_cols:
    print(col, check_outlier(df, col))

In [ ]:

def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit


for col in num_cols:
    replace_with_thresholds(df, col)


for col in num_cols:
    print(col, check_outlier(df, col))

In [ ]:
df.head()
import datetime as dt
# Preparation of CLTV metrics

df["Invoice_Date"].max()   # 2021-12-09

analysis_date = dt.datetime(2021,12, 10)

cltv = pd.DataFrame()

cltv = df.groupby("Customer_ID").agg({"Amount" : lambda x : x.sum(),
                             "Invoice_ID": lambda x : x.nunique(),
                             "Invoice_Date" : [lambda x: (analysis_date - x.min()).days,
                                              lambda x: (x.max() - x.min()).days] })
cltv.columns = cltv.columns.droplevel(0)

cltv.head()

In [ ]:
cltv.columns = ['monetary','frequency', 'T', 'recency' ]
cltv.head()

In [ ]:
# average earnings per purchase

cltv["monetary"] = cltv["monetary"] / cltv["frequency"]
cltv.head()

In [ ]:
cltv.describe().T

cltv = cltv[(cltv['frequency'] > 1)]

cltv.head()

In [ ]:
# we change recency & T (Customer's age)  value from day to Weekly

cltv["recency"] = cltv["recency"] / 7

cltv["T"] = cltv["T"] / 7

cltv.head()


In [ ]:

from lifetimes import BetaGeoFitter
from lifetimes import GammaGammaFitter
from lifetimes.plotting import plot_period_transactions
from scipy.special import logsumexp
from scipy.misc import logsumexp

# Establishment of BG-NBD Model and fit it

bgf = BetaGeoFitter(penalizer_coef=0.001)

bgf.fit(cltv['frequency'],
        cltv['recency'],
        cltv['T'])

In [ ]:
# We estimate expected purchases from customers within 3 months

bgf.predict(4 * 3,
            cltv['frequency'],
            cltv['recency'],
            cltv['T'])

cltv["exp_sales_3_month"] = bgf.predict(4 * 3,
                                               cltv['frequency'],
                                               cltv['recency'],
                                               cltv['T'])

cltv.head(10)

In [ ]:
# We estimate expected purchases from customers within 6 months

bgf.predict(4 * 6,
            cltv['frequency'],
            cltv['recency'],
            cltv['T'])

cltv["exp_sales_6_month"] = bgf.predict(4 * 6,
                                               cltv['frequency'],
                                               cltv['recency'],
                                               cltv['T'])


cltv.head(10)

In [ ]:
#  Establishment of Gamma-Gamma model and fit it


ggf = GammaGammaFitter(penalizer_coef=0.01)


ggf.fit(cltv['frequency'], cltv['monetary'])


In [ ]:
# customers expected average profit value


cltv["expected_average_profit"] = ggf.conditional_expected_average_profit(cltv['frequency'],
                                        cltv['monetary']).sort_values(ascending=False)

cltv.sort_values("expected_average_profit", ascending=False).head(10)




In [ ]:
# 6-month CLTV forecast

cltv_c = ggf.customer_lifetime_value(bgf,
                                   cltv['frequency'],
                                   cltv['recency'],
                                   cltv['T'],
                                   cltv['monetary'],
                                   time=6,
                                   discount_rate=0.01)


cltv["cltv"] = cltv_c

cltv.sort_values(by="cltv", ascending=False).head(10)

In [ ]:
# We divide all customers into 4 segments based on 6-month CLTV


cltv["segment"] = pd.qcut(cltv["cltv"], 4, labels=["D", "C", "B", "A"])

cltv.sort_values(by="cltv", ascending=False).head()

In [ ]:
cltv.groupby("segment").agg(
    {"count", "mean", "sum"})